# SCBE Canonical Colab Training Lane

One notebook for the real path: **raw generator export -> normalize -> QLoRA fine-tune**.

**Use this when:** you want one Colab lane instead of jumping between old copies.

**Guardrail:** do not continue past the preflight cell unless it passes.

**Expected flow:**
1. Generate a raw JSON export from the Spiralverse Protocol generator.
2. Upload that export here.
3. Normalize it into SCBE chat/SFT records.
4. Fine-tune on free Colab T4.
5. Optionally push the adapter to Hugging Face.


In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers>=4.44.0 datasets peft>=0.12.0 trl>=0.9.0 bitsandbytes>=0.43.0 accelerate>=0.33.0 huggingface_hub

In [ ]:
# Cell 2: Login to Hugging Face
from huggingface_hub import login
import os

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if not hf_token:
    hf_token = input('Enter your HuggingFace token (or leave blank to skip push): ').strip()

if hf_token:
    login(token=hf_token)
    print('Logged in to Hugging Face.')
else:
    print('Continuing without HF login. Training can still run locally in Colab.')


In [ ]:
# Cell 3: Runtime preflight (hard fail before long training)
import json
from pathlib import Path

import torch

failures = []
print('Running canonical Colab preflight...')
print(f'CUDA available: {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    failures.append('No GPU detected. Switch runtime to T4 GPU before training.')
else:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}')
    print(f'VRAM: {gpu_vram_gb:.1f} GB')

try:
    artifact_path = Path('/content/scbe_canonical_preflight.json')
    artifact_path.write_text(json.dumps({'failures': failures}, indent=2), encoding='utf-8')
    print(f'Artifact write: ok -> {artifact_path}')
except Exception as e:
    failures.append(f'Artifact write failed: {e}')

if failures:
    raise RuntimeError('Canonical Colab preflight failed:\n- ' + '\n- '.join(failures))

print('Canonical Colab preflight passed. Safe to continue.')


In [ ]:
# Cell 4: Upload raw generator export and normalize to chat records
import json
from pathlib import Path

from datasets import Dataset

from google.colab import files

print('Upload a raw Spiralverse generator JSON export...')
uploaded = files.upload()
raw_path = Path(next(iter(uploaded.keys())))

raw_payload = json.loads(raw_path.read_text(encoding='utf-8'))
conversations = raw_payload.get('conversations', []) if isinstance(raw_payload, dict) else []
if not conversations:
    raise RuntimeError('No conversations found. Expected a Spiralverse generator export with a top-level conversations array.')

records = []
for convo in conversations:
    turns = convo.get('turns', [])
    if len(turns) < 2:
        continue
    topic = convo.get('starting_topic', 'unknown')
    metadata = convo.get('metadata', {}) or {}
    lang = ''
    messages = []
    for idx, turn in enumerate(turns):
        content = str(turn.get('message', '')).strip()
        if not content:
            continue
        role = 'user' if idx % 2 == 0 else 'assistant'
        messages.append({'role': role, 'content': content})
        lang = lang or str(turn.get('language', '')).strip()
    if len(messages) < 2:
        continue
    formatted = []
    for msg in messages:
        formatted.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>")
    text = '\n'.join(formatted)
    records.append({
        'messages': messages,
        'text': text,
        'topic': topic,
        'language': lang,
        'metadata': json.dumps({
            'conversation_id': convo.get('id', ''),
            'starting_topic': topic,
            'generator_metadata': metadata,
        })
    })

if not records:
    raise RuntimeError('Normalization produced zero usable chat records.')

dataset = Dataset.from_list(records).shuffle(seed=42)
split = dataset.train_test_split(test_size=0.1, seed=42)

out_dir = Path('/content/scbe_canonical_artifacts')
out_dir.mkdir(parents=True, exist_ok=True)
train_path = out_dir / 'train_chat.jsonl'
val_path = out_dir / 'val_chat.jsonl'

with train_path.open('w', encoding='utf-8') as f:
    for row in split['train']:
        f.write(json.dumps(row, ensure_ascii=True) + '\n')
with val_path.open('w', encoding='utf-8') as f:
    for row in split['test']:
        f.write(json.dumps(row, ensure_ascii=True) + '\n')

print(f"Normalized records: {len(dataset)}")
print(f"Train: {len(split['train'])}, Val: {len(split['test'])}")
print(f"Artifacts: {train_path} and {val_path}")
print(split['train'][0]['text'][:300] + '...')


In [ ]:
# Cell 5: Load model and configure QLoRA
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = '/content/scbe-canonical-qwen-0.5b'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Cell 6: Train and optionally push adapter
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_seq_length=512,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    dataset_text_field='text',
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    processing_class=tokenizer,
)

trainer.train()
metrics = trainer.evaluate()
print(f"Final eval loss: {metrics['eval_loss']:.4f}")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved adapter to {OUTPUT_DIR}')

HF_REPO = 'issdandavis/scbe-canonical-colab-qwen-0.5b-lora'
if hf_token:
    model.push_to_hub(HF_REPO, commit_message='Canonical Colab lane adapter update')
    tokenizer.push_to_hub(HF_REPO, commit_message='Canonical Colab lane tokenizer')
    print(f'Pushed adapter to https://huggingface.co/{HF_REPO}')
else:
    print('HF push skipped because no token was provided.')
